# Week 9: Image Processing and Enhancement

## Topics

1. Review: Thai Flag, RGB/BGR, and Color Channels
2. Grayscale Conversion, Binary Images, and Thresholding
3. Brightness and Contrast: Dark, Bright, Low-Contrast, and High-Contrast Images
4. Reading Histograms Alongside Images
5. Intensity Transformations $s=T(r)$: Brightness, Linear Contrast, Negative, Gamma, and Gray-Level Slicing
6. Contrast Stretching and Histogram Equalization
7. Spatial Filters: Filter vs. Kernel and Noise Reduction
8. Kernels: Patch Slicing, Calculation, and Sliding
9. Changing Kernels: Sharpen and Sobel
10. Random Kernels and Multiple Feature Maps

พูดเรื่องการ ใช้ Grayscale หรือ Binary Images ในการทำ AI หรือหุ่นยนต์ ในการ ตรวจจับ แล้วแสดงเป็น RGB

## Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasuprint/AI-CE564/blob/main/weeks/W09_ImageProcessing/week9_colab.ipynb)

เลือก File → Save a copy in Drive หากต้องการเก็บงานของตัวเอง สำเนานี้เริ่มโดยไม่มี Output ให้รัน Setup แล้วรันแต่ละ Cell เพื่อสร้างผลภาพ ภาพประกอบใน Markdown โหลดจาก GitHub จึงต้องเชื่อมต่ออินเทอร์เน็ต


## 0. Setup

รัน Cell นี้ก่อน แล้วเลือก Runtime → Run all ได้เลย ระบบดาวน์โหลดภาพตัวอย่างจาก GitHub อัตโนมัติ ไม่ต้องอัปโหลดรูปหรือเชื่อม Google Drive


In [ ]:
# Colab มี NumPy, OpenCV และ Matplotlib พร้อมใช้แล้ว ไม่ต้อง Mount Google Drive
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import cv2
import matplotlib.pyplot as plt

BASE_URL = 'https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/'
image_dir = Path('week9_images')
image_dir.mkdir(exist_ok=True)
image_files = ['grace_hopper.jpg', 'camera.png', 'camera_demo_base.png', 'moon_demo_base.png', 'grace_noisy_1.png', 'grace_noisy_2.png', 'grace_noisy_3.png', 'grace_noisy_4.png', 'vintage_paper.png']

for filename in image_files:
    target = image_dir / filename
    if not target.is_file():
        urlretrieve(BASE_URL + filename, target)
    if cv2.imread(str(target)) is None:
        raise ValueError(f'อ่านไฟล์ภาพไม่ได้: {target} กรุณาลบไฟล์นี้แล้วรัน Setup ใหม่')

image_path = image_dir / 'grace_hopper.jpg'
plt.rcParams['figure.figsize'] = (6, 4)
print('Images ready:', len(image_files))
print('OpenCV:', cv2.__version__)
print('NumPy:', np.__version__)


## 1. Pixels and RGB Channels

ภาพสีเก็บเป็น Array ขนาด `(H, W, 3)` แต่ละพิกเซลมีสามค่า

- RGB เรียงค่า Red, Green, Blue ส่วน OpenCV อ่านภาพสีเป็น BGR
- แต่ละ Channel ของภาพ 8-bit มีค่า 0 ถึง 255
- NumPy อ่านค่าด้วย `image[y, x]` และเลือกแต่ละ Channel ด้วย `image[y, x, c]`


### 1.1 RGB Pixels and Coordinates

<img src="https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/rgb_pixel_coordinates.png" alt="ภาพสีแยกเป็นตาราง Red Green และ Blue โดยตำแหน่งเดียวกันเก็บค่าสามค่า" width="680" style="max-width:100%;height:auto">

TODO1:

ภาพ RGB ตัวอย่างนี้มี \_**_ แถว _** คอลัมน์ รวม **_ พิกเซล แต่ละพิกเซลเก็บ _** ค่า จึงมี Shape ของ Image เป็น `(2, 3, 3)` ดังนั้นจึงเก็บตัวเลขทั้งหมด 18 ค่า จุด (0, 0)อยู่มุมซ้ายบน โดย x เพิ่มไปทางขวา และ y เพิ่มลงด้านล่าง

พิกเซลสีเขียวเข้มอยู่ที่ `(x=1, y=1)` อ่านด้วย `image[1, 1]` ได้ค่า RGB `[11, 102, 35]` ถ้าต้องการเฉพาะค่า Green ใช้ `image[1, 1, __]` จะได้ \_\_
โดย Index ตัวสุดท้ายเลือก Channel: 0 คือ Red, 1 คือ Green และ 2 คือ Blue


### 1.2 ตำแหน่ง Pixel และค่าของแต่ละ Channel

<img src="https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/RGBImagge.png" alt="ภาพสีแยกเป็นตาราง Red Green และ Blue โดยตำแหน่งเดียวกันเก็บค่าสามค่า" width="680" style="max-width:100%;height:auto">

ภาพสี (RGB) Red, Green และ Blue ขนาดเท่ากัน ที่พิกัดเดียวกันจึงมีสามค่า เช่น ตัวอย่างในรูปใช้ RGB = `(11, 102, 35)` หนึ่งพิกเซลย ณ ตำแหน่งเดียว จะมีข้อมูลสาม Channel มี Shape `(H, W, 3)`


### 1.3 RGB Values and Color

<img src="https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/Color.png" alt="ตัวอย่างสี Forest Green จากค่า Red 11 Green 102 และ Blue 35" width="460" style="max-width:100%;height:auto">

ค่า RGB `(11, 102, 35)` ให้สีเขียวตามตัวอย่าง เพราะค่า Green สูงกว่าอีกสอง Channel เครื่องหมายบวกในรูปสื่อถึงการผสมแสงสามสี (การซ้อนทับ)เป็นการซ้อนกันของความเข้มของสีแต่ละ Channel


In [ ]:
flag_rgb = np.zeros((300, 600, 3), dtype=np.uint8)

# Show the flag_rgb initial black image
plt.imshow(flag_rgb)
plt.title('Thai flag: RGB array')
plt.axis('off')
plt.show()

flag_rgb[0:50, :] = (255, 0, 0)       # แดง
flag_rgb[50:100, :] = (255, 255, 255) # ขาว
flag_rgb[100:200, :] = (0, 0, 255)    # น้ำเงิน
flag_rgb[200:250, :] = (255, 255, 255)
flag_rgb[250:300, :] = (255, 0, 0)

print('Shape:', flag_rgb.shape)
print('Red pixel (RGB):', flag_rgb[0, 0])
print('Pixels:', 300 * 600)
print('Stored values:', flag_rgb.size)
plt.imshow(flag_rgb)
plt.title('Thai flag: RGB array')
plt.axis('off')
plt.show()

### 1.4 Splitting RGB Channels

`[:, :, 0]` เลือก Channel 0 โดยเลือก Value จากทุกแถวและทุกคอลัมน์ สำหรับภาพ RGB เป็นการนำค่า Channer \_\_\_ (Red, Green, Blue)

แสดงความเข้มแต่ละ Channel ด้วยสีเทา (Intensity ) ตำแหน่งที่สีขาว แปลว่าค่าความเข้ม ของ Channel นั้นสูง ส่วนตำแหน่งที่สีดำ แปลว่าค่าความเข้มต่ำ เนื่องจากเราแสดงระดับเป็นสีดำถึงขาว (สิ่งที่แสดงมีใว้เพื่อการ Visualize) แต่ตอนแสดงภาพเราไม่ได้ใช้ Grayscale แต่เราใช้ความเข้มแต่ละ channel มาเรียงกันเป็น RGB


In [ ]:
red = flag_rgb[:, :, 0]
green = flag_rgb[:, :, 1]
blue = flag_rgb[:, :, 2]

channels = [red, green, blue]
titles = ['Red channel', 'Green channel', 'Blue channel']
plt.figure(figsize=(12, 3))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(channels[i], cmap='gray', vmin=0, vmax=255)
    plt.title(titles[i])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Grayscale and Binary Images

### Grayscale Conversion

Grayscale Conversion คือกระบวนการแปลงภาพสีจากการแทนข้อมูลสาม Channel ได้แก่ Red, Green และ Blue ให้เป็นภาพความเข้มหนึ่ง Channel โดยคำนวณผลรวมเชิงเส้นแบบถ่วงน้ำหนักของค่า RGB ดังสมการโดยประมาณ

$$Y\approx0.299R+0.587G+0.114B$$

โดย $R$, $G$ และ $B$ คือค่าของแต่ละ Channel ณ ตำแหน่งพิกเซลเดียวกัน และ $Y$ คือค่าความเข้มที่ได้ สัมประสิทธิ์ดังกล่าวให้น้ำหนักองค์ประกอบสีเขียวสูงที่สุด เพื่อประมาณความสว่างตามการรับรู้ของระบบการมองเห็นมนุษย์

สำหรับภาพ Grayscale แบบ 8-bit ค่าความเข้มอยู่ในช่วง 0 ถึง 255 โดย 0 แทนสีดำ และ 255 แทนสีขาว การแปลงคงความละเอียดเชิงพื้นที่เดิม (Spatial) แต่เปลี่ยน Shape จาก `(H, W, 3)` เป็น `(H, W)` ข้อมูลสีที่สูญเสียไปทำให้ไม่สามารถกู้ค่า RGB เดิมจากค่าความเข้มเพียงอย่างเดียวได้


In [ ]:
# 1. Read the image using OpenCV
image_bgr = cv2.imread(str(image_path))

# 2. Convert BGR to RGB for displaying with Matplotlib
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# 3. Convert RGB to Grayscale
gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)


# TODO2: ลองใช้ Equation 1.1 แปลงภาพ RGB เป็น Grayscale 

# 4. Convert RGB to Grayscale manually using the formula. 
# (Solution)
gray_manual = 0.299 * image_rgb[:, :, 0] + 0.587 * image_rgb[:, :, 1] + 0.114 * image_rgb[:, :, 2]


# 4. Display the original RGB image and the Grayscale image
plt.figure(figsize=(10, 4))
plt.subplot(1, 3, 1)
plt.imshow(image_rgb)
plt.title('Original RGB Image')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(gray, cmap='gray')
plt.title('Grayscale Image')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(gray_manual, cmap='gray')
plt.title('Manual Grayscale Image')
plt.axis('off')
plt.tight_layout()
plt.show()

### 2.2 Binary Images and Thresholding

Binary Image คือภาพที่แต่ละพิกเซลมีค่าได้เพียงสองระดับ โดยทั่วไปแทนด้วย 0 และ 1 หรือใช้ค่า 0 และ 255 เมื่อจัดเก็บเป็นภาพชนิด `uint8` เพื่อแสดงสีดำและสีขาวตามลำดับ แม้มีข้อมูลเพียงสองระดับ การจัดเก็บในรูปแบบนี้ยังใช้ 8 bits ต่อพิกเซล

การแปลงภาพ Grayscale เป็น Binary Image ด้วย Thresholding เป็นการจำแนกพิกเซลตามค่าความเข้ม โดยกำหนดค่า Threshold $T$ เป็นเกณฑ์ ดังสมการ

$$B(x,y)=\begin{cases}255&I(x,y)>T\\0&I(x,y)\leq T\end{cases}$$

โดย $I(x,y)$ คือค่าความเข้มของภาพต้นฉบับ และ $B(x,y)$ คือค่าพิกเซลของภาพผลลัพธ์ ณ ตำแหน่งเดียวกัน การใช้ $T$ ค่าเดียวทั่วทั้งภาพเรียกว่า Global Thresholding ภายใต้เงื่อนไขนี้ การเพิ่ม $T$ ทำให้จำนวนพิกเซลสีขาวลดลงหรือคงเดิม


งานที่นิยมนำ Thresholding ไปใช้ เช่น การแยกวัตถุออกจากพื้นหลัง (Segmentation) หรือการสร้าง Binary Mask เพื่อระบุบริเวณที่สนใจในภาพ
Thresholding ใช้สร้าง Binary Mask เพื่อระบุบริเวณที่ผ่านเกณฑ์ความเข้ม แต่ไม่ได้จำแนกชนิดวัตถุโดยตรง หากวัตถุและพื้นหลังมีความเข้มใกล้เคียงกัน หรือแสงสว่างไม่สม่ำเสมอ การใช้ Threshold เดียวอาจแยกบริเวณได้ไม่ครบถ้วน


In [ ]:
gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

# 5. Apply binary thresholding T = 127
binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)[1]

# 6. Display the Grayscale and Binary images
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(gray, cmap='gray')
plt.title('Grayscale Image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(binary, cmap='gray')
plt.title('Binary Image (Threshold = 127)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Brightness and Contrast

### 3.1 Brightness

- นิยาม: ระดับความมืดหรือสว่างที่ปรากฏในภาพ Brightness สูงทำให้ภาพสว่างขึ้น และ Brightness ต่ำทำให้ภาพมืดลง
- การปรับ: การบวกค่าคงที่ $\beta$ ให้ทุกพิกเซลตาม $s=r+\beta$ ทำให้ภาพสว่างขึ้นเมื่อ $\beta>0$ และมืดลงเมื่อ $\beta<0$
- ผลต่อรายละเอียด: ความต่างของความเข้มระหว่างพิกเซลจะคงเดิม การเพิ่ม Brightness จึงไม่ได้เพิ่ม Contrast หรือข้อมูลรายละเอียดโดยตรง

### 3.2 Contrast

- นิยาม: ระดับความแตกต่างของความเข้มระหว่างบริเวณในภาพ Contrast สูงทำให้ส่วนมืดและส่วนสว่างแตกต่างกันมาก ส่วน Contrast ต่ำทำให้ค่าความเข้มใกล้กันและรายละเอียดอาจกลืนกัน
- การปรับ: การคูณความต่างจากค่าอ้างอิง $c$ ตาม $s=\alpha(r-c)+c$ จะขยายความต่างเมื่อ $\alpha>1$ และบีบความต่างเมื่อ $0<\alpha<1$ โดยพิจารณาก่อนเกิด Clipping
- ผลต่อความสว่าง: การปรับ Contrast อาจเปลี่ยนค่าเฉลี่ยด้วย หากเลือก $c$ เท่ากับค่าเฉลี่ยเดิม ค่าเฉลี่ยจะคงเดิมก่อน Clipping และการปัดเศษ
- การประเมิน: ควรแยก Global Contrast ซึ่งพิจารณาภาพโดยรวม ออกจาก Local Contrast ซึ่งพิจารณาความต่างในบริเวณเฉพาะ ช่วง Min–Max เพียงอย่างเดียวไม่เพียงพอสำหรับประเมินรายละเอียดทั้งหมด

Brightness และ Contrast เป็นคุณลักษณะของภาพที่สัมพันธ์กัน แต่ใช้อธิบายคนละด้าน ภาพอาจมี Brightness ต่ำและ Contrast สูง หรือมี Brightness สูงและ Contrast ต่ำได้ การประเมินจึงควรพิจารณาทั้งสองลักษณะร่วมกับรายละเอียดที่ต้องการวิเคราะห์

### 3.3 Effects, Benefits, and Limitations

ตารางนี้อธิบายผลของการเพิ่มหรือลดจากภาพตั้งต้นเดียวกัน ระดับที่เหมาะสมขึ้นอยู่กับภาพและวัตถุประสงค์ **ไม่ได้หมายความว่าค่าสูงกว่าจะให้ผลดีกว่าเสมอ**

| การปรับ          | ผลที่สังเกตได้                               | ข้อดีเมื่อเลือกใช้อย่างเหมาะสม                                             | ข้อเสียหรือข้อจำกัด                                                                               |
| ---------------- | -------------------------------------------- | -------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------- |
| เพิ่ม Brightness | ค่าความเข้มโดยรวมสูงขึ้น ภาพสว่างขึ้น        | ช่วยให้บริเวณมืดที่ยังพอมีข้อมูลที่สามารถมองเห็น สามารถมองเห็นได้สะดวกขึ้น | ส่วนสว่างอาจถูก Clip เป็น 255 และสูญเสียรายละเอียด การบวกค่าคงที่ไม่ได้เพิ่มความต่างระหว่างพิกเซล |
| ลด Brightness    | ค่าความเข้มโดยรวมต่ำลง ภาพมืดลง              | ลดความสว่างในการแสดงผล หรือใช้สร้างลักษณะภาพโทนมืด                         | ส่วนมืดอาจถูก Clip เป็น 0 และรายละเอียดมองยากขึ้น **ไม่สามารถกู้รายละเอียดที่ขาวอิ่มตัวไปแล้ว**   |
| เพิ่ม Contrast   | ความต่างจากค่าอ้างอิงมากขึ้น                 | ช่วยแยกบริเวณที่มีความเข้มใกล้กัน และเน้นรายละเอียดที่มีอยู่               | Noise อาจเด่นขึ้น ส่วนมืดหรือสว่างอาจถูก Clip จนรายละเอียดหาย และภาพอาจดูแข็งเกินไป               |
| ลด Contrast      | ความต่างจากค่าอ้างอิงน้อยลง ภาพดูนุ่มหรือซีด | ลดความจัดของภาพ หรือใช้สร้างลักษณะ Fade และ Retro                          | ขอบและรายละเอียดอาจกลืนกัน การแยกวัตถุจากพื้นหลังด้วยความเข้มอาจยากขึ้น                           |

**Clipping คือการจำกัดค่าที่เกินช่วงให้เป็น 0 หรือ 255 การปรับ Brightness หรือ Contrast ไม่สามารถกู้ข้อมูลที่สูญเสียจาก Clipping ไปแล้วได้**


In [ ]:
camera = cv2.imread(str(image_dir / 'camera.png'), cv2.IMREAD_GRAYSCALE)

demo_base = cv2.imread(str(image_dir / 'camera_demo_base.png'), cv2.IMREAD_GRAYSCALE)


base_float = demo_base.astype(np.float32)
base_mean = float(base_float.mean())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, image, title in zip(axes, [camera, demo_base], ['Original Camera', 'Demo base: prepared image']):
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# 1. Brightness: บวก/ลบค่าคงที่ ความต่างระหว่างพิกเซลยังเท่าเดิม
dark = np.round(base_float - 80).astype(np.uint8)
bright = np.round(base_float + 80).astype(np.uint8)
brightness_images = [dark, demo_base, bright]
brightness_titles = ['Low brightness (-80)', 'Demo base', 'High brightness (+80)']

plt.figure(figsize=(12, 4))
for i, (image, title) in enumerate(zip(brightness_images, brightness_titles)):
    plt.subplot(1, 3, i + 1)
    plt.imshow(image, cmap='gray', vmin=0, vmax=255)
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Brightness: บวก/ลบค่าคงที่ ความต่างระหว่างพิกเซลยังเท่าเดิม
dark = np.clip(np.round(base_float - 40), 0, 255).astype(np.uint8)

bright = np.clip(np.round(base_float + 40), 0, 255).astype(np.uint8)

brightness_images = [dark, demo_base, bright]
brightness_titles = ['Low brightness (-40)', 'Demo base', 'High brightness (+40)']

# 2. Show the images and their histograms
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for i, (image, title) in enumerate(zip(brightness_images, brightness_titles)):
    axes[0, i].imshow(image, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(title)
    axes[0, i].axis('off')
    ax = axes[1, i]
    ax.hist(image.ravel(), bins=32, range=(0, 256), color='gray')
    ax.axvline(image.mean(), color='darkorange', linestyle='--', label='Mean')
    ax.set_xlim(0, 256)
    ax.set_xlabel('Intensity (32 bins, 8 levels per bin)')
    ax.set_ylabel('Pixel count')
    ax.set_title(f'Mean={image.mean():.1f} | Min={image.min()}, Max={image.max()}')
    ax.legend()

# ใช้สเกลแกน Y ในการเปรียบเทียบ Scale
brightness_ymax = max(ax.get_ylim()[1] for ax in axes[1])
for ax in axes[1]:
    ax.set_ylim(0, brightness_ymax)
fig.suptitle('Brightness: the histogram shifts; the intensity range stays the same')
plt.tight_layout()
plt.show()


### Contrast Adjustment Around the Mean

การปรับ Contrast ในตัวอย่างนี้ใช้ค่าเฉลี่ยความเข้มของภาพตั้งต้นเป็นจุดอ้างอิง โดยกำหนด

$$\mu=\frac{1}{N}\sum_{i=1}^{N}r_i,\qquad s=\alpha(r-\mu)+\mu$$

- $r$ คือค่าความเข้มเดิมของพิกเซล ตรงกับค่าใน `base_float`
- $\mu$ คือค่าเฉลี่ยของภาพตั้งต้น ตรงกับ `base_mean` และใช้ค่าเดียวกันกับทุกพิกเซล
- $\alpha$ คือตัวคูณสำหรับปรับ Contrast
- $s$ คือค่าความเข้มหลังการปรับ ก่อนปัดเศษหรือจำกัดช่วงค่า

จากสมการ $s-\mu=\alpha(r-\mu)$ ตัวคูณจึงกำหนดระยะห่างของค่าพิกเซลจากค่าเฉลี่ย

| ตัวคูณ       | สมการ              | ผลต่อภาพและ Histogram                                        |
| ------------ | ------------------ | ------------------------------------------------------------ |
| $\alpha=0.5$ | $s=0.5(r-\mu)+\mu$ | บีบค่าเข้าหาค่าเฉลี่ย ความต่างลดลง Histogram แคบลง           |
| $\alpha=1$   | $s=r$              | ค่าคงเดิม เป็นภาพอ้างอิง                                     |
| $\alpha=1.5$ | $s=1.5(r-\mu)+\mu$ | ขยายค่าออกจากค่าเฉลี่ย ความต่างเพิ่มขึ้น Histogram กว้างขึ้น |


In [ ]:
# 2. Contrast: บีบ/ขยายค่า  Histrogram รอบๆ ค่าเฉลี่ยเดิม 

base_float = demo_base.astype(np.float32)
base_mean = float(base_float.mean())

low_contrast = np.round(0.5 * (base_float - base_mean) + base_mean).astype(np.uint8)
high_contrast = np.round(1.5 * (base_float - base_mean) + base_mean).astype(np.uint8)
contrast_images = [low_contrast, demo_base, high_contrast]
contrast_titles = ['Low contrast (x0.5)', 'Demo base', 'High contrast (x1.5)']

# Show the images and their histograms
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey='row')

for i in range(3):
    image = contrast_images[i]

    axes[0, i].imshow(image, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(contrast_titles[i])
    axes[0, i].axis('off')

    axes[1, i].hist(image.ravel(), bins=32, range=(0, 256), color='gray')
    axes[1, i].axvline(image.mean(), color='darkorange', linestyle='--')
    axes[1, i].set_title(f'Mean = {image.mean():.1f}')
    axes[1, i].set_xlim(0, 256)
    axes[1, i].set_xlabel('Intensity')

axes[1, 0].set_ylabel('Pixel count')
plt.tight_layout()
plt.show()


In [ ]:
# 3.  LC/LB, HC/LB, LC/HB, HC/HB

combined_settings = [(0.5, -40), (1.5, -40), (0.5, 40), (1.5, 40)]

combined_titles = ['Low brightness / Low contrast', 
                    'Low brightness / High contrast',
                    'High brightness / Low contrast', 'High brightness / High contrast']
combined_images = []
fig, axes = plt.subplots(4, 2, figsize=(11, 13))
hist_axes = []

# ตำแหน่งภาพทั้งสี่: (แถว 0, ซ้าย/ขวา) และ (แถว 2, ซ้าย/ขวา)
image_rows = [0, 0, 2, 2]
image_cols = [0, 1, 0, 1]

for i in range(4):
    alpha = combined_settings[i][0]
    beta = combined_settings[i][1]
    title = combined_titles[i]
    values = alpha * (base_float - base_mean) + base_mean + beta
    # ตรวจค่าก่อนแปลงเป็น uint8 เพื่อไม่ให้เลขวนกลับหรือมี Clipping
    assert values.min() >= 0 and values.max() <= 255
    image = np.round(values).astype(np.uint8)
    combined_images.append(image)
    row = image_rows[i]
    col = image_cols[i]
    axes[row, col].imshow(image, cmap='gray', vmin=0, vmax=255)
    axes[row, col].set_title(title)
    axes[row, col].axis('off')
    ax = axes[row + 1, col]
    ax.hist(image.ravel(), bins=32, range=(0, 256), color='gray')
    ax.axvline(image.mean(), color='darkorange', linestyle='--', label='Mean')
    ax.set_xlim(0, 256)
    ax.set_xlabel('Intensity (32 bins, 8 levels per bin)')
    ax.set_ylabel('Pixel count')
    ax.set_title(f'Mean={image.mean():.1f} | Min={image.min()}, Max={image.max()}')
    ax.legend()
    hist_axes.append(ax)

combined_ymax = max(ax.get_ylim()[1] for ax in hist_axes)

for ax in hist_axes:
    ax.set_ylim(0, combined_ymax)
fig.suptitle('Brightness and contrast: four combinations from the same base')
plt.tight_layout()
plt.show()

### Discussion: Brightness and Contrast (TODOs 3:)

1. ภาพ Dark กับ Low contrast ต่างกันตรงไหนบนกราฟ?
2. ทำไมภาพ Low contrast มียอดสูงได้?
3. เมื่อภาพ Bright สว่างขึ้น ความต่างระหว่างพิกเซลมากขึ้นด้วยไหม?
4. ถ้า Original เห็นรายละเอียดดีอยู่แล้ว จำเป็นต้องปรับต่อไหม? เพราะอะไร?


## 5. Intensity Transformations

### The Mapping: $s=T(r)$

$r$ คือค่าความเข้มเดิม $T$ คือกฎการแปลง และ $s$ คือค่า Pixel ใหม่

### Data Types and Clipping

คำนวณด้วย Float ก่อน แล้วจำกัดค่าด้วย Clipping ให้อยู่ระหว่าง 0 ถึง 255 ก่อนแปลงกลับเป็น `uint8` ค่าที่ถูก Clip รวมกันที่ขอบจะเสียรายละเอียดความต่างเดิม

### Image Negative

Image Negative เป็นการกลับระดับความเข้มแต่ละ Pixel โดยเปลี่ยนบริเวณมืดเป็นสว่างและบริเวณสว่างเป็นมืด สำหรับภาพที่มีระดับความเข้ม $L$ ระดับ กำหนด Mapping เป็น

$$s=(L-1)-r$$

เมื่อใช้ภาพ 8-bit จะมี $L=256$ จึงได้

$$s=255-r$$

![Image Negative: original and inverted intensity](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/ImageNegative.png)

ภาพประกอบแสดงภาพเต้านมก่อนและหลังกลับระดับความเข้ม การกลับค่าอาจช่วยให้บางรายละเอียดสังเกตได้ง่ายขึ้นตามรูปแบบการแสดงผล

### Log Transformation

Log Transformation เป็นการแปลงแบบไม่เป็นเส้นตรงที่ขยายความต่างในช่วงค่าต่ำและบีบความต่างในช่วงค่าสูง เหมาะกับการแสดงข้อมูลที่มี Dynamic Range กว้าง เช่น ขนาดของ Fourier Spectrum

$$s=c\ln(1+r),\qquad r\geq0,\ c>0$$

การบวก 1 ทำให้คำนวณที่ $r=0$ ได้ หากต้องการ Mapping ช่วง $[0,r_{max}]$ ไปยัง $[0,255]$ สามารถกำหนด

$$c=\frac{255}{\ln(1+r_{max})},\qquad r_{max}>0$$

![Log Transformation: weak patterns become more visible](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/LogTransform.png)

ภาพประกอบแสดงรูปแบบที่มีจุดสว่างเด่นตรงกลางและลวดลายจางรอบนอก หลังแปลงจะเห็นลวดลายค่าต่ำได้ชัดขึ้น โดยลดความแตกต่างเชิงสัดส่วนระหว่างบริเวณค่าต่ำกับค่าสูง การแปลงช่วยแสดงข้อมูลที่มีอยู่ แต่ก็ทำให้ Noise ในช่วงค่าต่ำอาจเด่นขึ้นด้วย

### Power-Law (Gamma) Transformation

Power-Law Transformation ปรับความเข้มด้วยเลขชี้กำลัง $\gamma$ สำหรับค่าที่ Normalize ให้อยู่ในช่วง 0 ถึง 1 ใช้รูปทั่วไป $v=c\,u^\gamma$ โดยในตัวอย่างกำหนด $c=1$ และ $u=r/255$ จึงเขียนสำหรับภาพ 8-bit ได้เป็น

$$s=255\left(\frac{r}{255}\right)^\gamma,\qquad \gamma>0$$

| ค่า Gamma    | ผลต่อระดับความเข้ม                        |
| ------------ | ----------------------------------------- |
| $0<\gamma<1$ | ค่าระหว่าง 0 และ 255 สูงขึ้น ภาพสว่างขึ้น |
| $\gamma=1$   | ค่าคงเดิม                                 |
| $\gamma>1$   | ค่าระหว่าง 0 และ 255 ต่ำลง ภาพมืดลง       |

![Power-Law Transformation: spinal images at different display intensities](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/PowerLaw.png)

Gamma เปลี่ยนแต่ละช่วงความเข้มไม่เท่ากัน โดยค่า 0 และ 255 ยังเป็นขอบบนและขอบล่างของค่า Pixel ต่างจากการเพิ่ม Brightness ด้วยค่าคงที่ เช่น เมื่อ $r=64$ และ $\gamma=0.5$ จะได้ $s\approx128$ แต่เมื่อ $r=225$ จะได้ $s\approx240$ ส่วนมืดจึงถูกเพิ่มค่ามากกว่าในตัวอย่าง


In [ ]:
# TODO4: ใช้ภาพ grace_hopper.jpg แสดงผลลัพธ์ของการปรับ Brightness และ Contrast ทั้งสี่แบบ

# 4.1 Image Negative: Invert pixel intensities

negative_gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
negative_image = 255 - negative_gray

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(negative_gray, cmap='gray', vmin=0, vmax=255)
plt.title('Original Grayscale')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(negative_image, cmap='gray', vmin=0, vmax=255)
plt.title('Image Negative: 255 - r')
plt.axis('off')

plt.tight_layout()
plt.show()


## 6. Contrast Stretching and Histogram Equalization

### 6.1 Contrast Stretching

ยืดค่าต่ำสุดและสูงสุดของภาพให้เป็น 0 และ 255 ด้วยสมการเส้นตรง

$$s=\frac{r-r_{min}}{r_{max}-r_{min}}\times255$$

$r$ คือค่าเดิม และ $s$ คือค่าใหม่ วิธีนี้ขยายช่วงค่า แต่พิกเซลส่วนใหญ่อาจยังกระจุกอยู่ หากทุกพิกเซลมีค่าเท่ากันต้องจัดการแยกเพื่อไม่ให้หารด้วยศูนย์

นิยมใช้ปรับภาพที่มีช่วงความเข้มแคบ เช่น ภาพที่ดูซีด เพื่อเพิ่ม Contrast และให้ส่วนมืดกับส่วนสว่างแยกกันชัดขึ้น

### 6.2 Histogram Equalization

ใช้จำนวนพิกเซลแต่ละระดับสร้าง Mapping ใหม่ผ่านสัดส่วนสะสม หรือ CDF ช่วยขยายความต่างในช่วงที่มีพิกเซลกระจุกตัว ใช้คำสั่ง `cv2.equalizeHist()` ได้โดยตรงกับภาพ Grayscale แบบ `uint8`

นิยมใช้เพิ่ม Contrast ในภาพที่พิกเซลส่วนใหญ่มีความเข้มใกล้กัน เพื่อให้เห็นพื้นผิวหรือรายละเอียดได้ง่ายขึ้น เช่น ภาพดวงจันทร์ แต่ Contrast อาจจัดเกินไปและ Noise อาจเด่นขึ้น


In [ ]:
moon_base = cv2.imread(str(image_dir / 'moon_demo_base.png'), cv2.IMREAD_GRAYSCALE)


# 1. Stretching: คำนวณด้วย Float ก่อนแปลงกลับเป็น uint8
moon_float = moon_base.astype(np.float32)
moon_min = float(moon_float.min())
moon_max = float(moon_float.max())
if moon_max > moon_min:
    moon_stretched = np.round((moon_float - moon_min) / (moon_max - moon_min) * 255).astype(np.uint8)
else:
    moon_stretched = moon_base.copy()

# 2. Equalization: เริ่มจาก moon_base เดียวกัน ใช้ Grayscale แบบ uint8
moon_equalized = cv2.equalizeHist(moon_base)

moon_images = [moon_base, moon_stretched, moon_equalized]
moon_titles = ['Prepared Moon: narrow range', 'Min-max stretching', 'Histogram equalization']

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey='row')
for i in range(3):
    image = moon_images[i]
    axes[0, i].imshow(image, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(moon_titles[i])
    axes[0, i].axis('off')

    axes[1, i].hist(image.ravel(), bins=32, range=(0, 256), color='gray')
    axes[1, i].set_xlim(0, 256)
    axes[1, i].set_xlabel('Intensity')
    axes[1, i].set_title(f'Min={image.min()}, Max={image.max()}, Mean={image.mean():.1f}')

axes[1, 0].set_ylabel('Pixel count')
plt.tight_layout()
plt.show()


### What to Observe

| ภาพ           | สิ่งที่สังเกต                                                          |
| ------------- | ---------------------------------------------------------------------- |
| Prepared Moon | Histogram แคบ รายละเอียดพื้นผิวกลืนกัน                                 |
| Stretching    | ยืดช่วงค่าเป็น 0–255 แต่พิกเซลส่วนใหญ่อาจยังกระจุก                     |
| Equalization  | กระจายค่าตามจำนวนพิกเซล รายละเอียดบางส่วนเด่นขึ้น แต่ภาพอาจดูจัดเกินไป |

ทั้งสองวิธีอาจทำให้ Noise เด่นขึ้น จึงควรเลือกจากรายละเอียดที่ต้องการดู

หมายเหตุ: หากภาพมี Min = 0 และ Max = 255 อยู่แล้ว Min–Max Stretching จะไม่เปลี่ยนภาพ


## 7. Spatial Filters: Reducing Noise

- **Filter:** วิธีประมวลผลภาพ เช่น ลด Noise หรือหาขอบ
- **Kernel:** ตารางน้ำหนักที่เลื่อนบนภาพเพื่อคำนวณผลจากพิกเซลรอบข้าง ในตัวอย่าง Linear Filtering นี้

### Mean Filter

เฉลี่ยพิกเซลเก้าจุดเพื่อลด Noise แต่ทำให้รายละเอียด Blur ด้วย

$$K_{mean}=\frac{1}{9}\begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}$$

```text
Noisy Image → Mean 3×3 → Smoothed Image
```

สังเกต Noise บนพื้นหลังและความคมของเครื่องแบบก่อน–หลัง


In [ ]:
filter_gray = cv2.imread(str(image_dir / 'grace_hopper.jpg'), cv2.IMREAD_GRAYSCALE)

filter_input = filter_gray.astype(np.float32)

# Noise จำลอง: Seed คงที่เพื่อให้ทดลองซ้ำได้
noise_rng = np.random.default_rng(9)
noise = noise_rng.normal(0, 30, filter_gray.shape)
noisy_image = np.clip(filter_input + noise, 0, 255).astype(np.float32)

mean_kernel = np.ones((3, 3), dtype=np.float32) / 9
mean_output = cv2.filter2D(noisy_image, cv2.CV_32F, mean_kernel)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(noisy_image, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Input: Gaussian noise')
axes[1].imshow(mean_kernel, cmap='Blues', vmin=0, vmax=1/9)
axes[1].set_title('Mean kernel 3x3')
for row in range(3):
    for col in range(3):
        axes[1].text(col, row, '1/9', ha='center', va='center', color='white')
axes[2].imshow(mean_output, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Output: smoother, but blurred')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 8. Kernels: Slice, Calculate, and Slide

เลือก Patch ด้วย `image[y:y+3, x:x+3]` คูณน้ำหนักตำแหน่งที่ตรงกัน แล้วบวกเป็น Output หนึ่งช่อง

$$
P=\begin{bmatrix}10&20&30\\60&70&80\\110&120&130\end{bmatrix},\qquad
s=\frac{10+20+30+60+70+80+110+120+130}{9}=70
$$

```text
Slice Patch → คูณ Kernel → บวก → เขียน Output → Slide
```

ใช้ Kernel เดิมและอ่าน Input เดิมทุกตำแหน่ง ตัวอย่างนี้เลื่อนทีละ 1 ช่อง (Stride = 1) ไม่เติมขอบ: Input 5×5 → Output 3×3


### Sliding Animation

สีส้มแสดง Patch ที่อ่านและช่อง Output ที่เขียน สีฟ้าแสดงจุดกลาง Patch

![Mean kernel sliding across nine positions](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/kernel_sliding.gif)

[ดูภาพนิ่งหาก GIF ไม่เล่น](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/kernel_sliding_steps.png)

ภาพเต็มใช้ `cv2.filter2D()` เติมขอบแบบค่าซ้ำ (`BORDER_REPLICATE`) จึงได้ Output ขนาดเดิม คำสั่งนี้ไม่กลับด้าน Kernel หรือเรียกว่า Cross-correlation


## 9. Changing the Kernel: Sharpen and Sobel

### 9.1 Sharpen

เพิ่มความต่างระหว่างจุดกลางกับรอบข้าง รายละเอียดจึงเด่นขึ้น แต่ Noise อาจเด่นตาม

$$K_{sharpen}=\begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}$$


In [ ]:
sharpen_kernel = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]], dtype=np.float32)

sharpen_raw = cv2.filter2D(filter_input, cv2.CV_32F, sharpen_kernel,
                          borderType=cv2.BORDER_REPLICATE)

sharpen_display = np.clip(sharpen_raw, 0, 255)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(filter_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Input: original grayscale')
axes[1].imshow(sharpen_kernel, cmap='coolwarm', vmin=-5, vmax=5)
axes[1].set_title('Sharpen kernel 3x3')
for row in range(3):
    for col in range(3):
        axes[1].text(col, row, f'{sharpen_kernel[row, col]:.0f}', ha='center', va='center')
axes[2].imshow(sharpen_display, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Output: sharpened (clipped for display)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


### 9.2 Sobel X

เปรียบเทียบความเข้ม บริเวณทางด้านซ้ายและขวา จึงตอบสนองต่อขอบแนวตั้ง ส่วนบริเวณความเข้มคงที่ให้ผลใกล้ศูนย์

$$K_x=\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}$$


In [ ]:
sobel_kernel = np.array([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]], dtype=np.float32)

sobel_raw = cv2.filter2D(filter_input, cv2.CV_32F, sobel_kernel,
                        borderType=cv2.BORDER_REPLICATE)

# เพื่อค่าติดลบ
sobel_strength = np.abs(sobel_raw)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(filter_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Input: original grayscale')
axes[1].imshow(sobel_kernel, cmap='coolwarm', vmin=-2, vmax=2)
axes[1].set_title('Sobel X kernel 3x3')
for row in range(3):
    for col in range(3):
        axes[1].text(col, row, f'{sobel_kernel[row, col]:.0f}', ha='center', va='center')
edge_plot = axes[2].imshow(sobel_strength, cmap='gray', vmin=0,
                            vmax=max(1.0, float(sobel_strength.max())))
axes[2].set_title('Output: absolute edge response')
fig.colorbar(edge_plot, ax=axes[2], fraction=0.046, pad=0.04)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

# ลองเปลี่ยน Kernel เป็น sobel_kernel.T เพื่อดู Sobel Y


### Summary

| Kernel  | ผลที่เห็น                       |
| ------- | ------------------------------- |
| Mean    | Noise ลดลง แต่ภาพ Blur          |
| Sharpen | รายละเอียดและ Noise อาจเด่นขึ้น |
| Sobel X | ขอบแนวตั้งเด่น                  |

วิธี Slide เหมือนเดิม เปลี่ยนน้ำหนักจึงได้ผลต่างกัน


## 10. Random Kernels and Feature Maps

### 10.1 One Random Kernel

สุ่ม Kernel 3×3 แล้ว Slide บน Grayscale ช่วง 0–1 เพื่อดูผลตอบสนองต่อขอบ ลวดลาย หรือความสว่าง


In [ ]:
feature_input = filter_input / 255.0
kernel_rng = np.random.default_rng(12)
random_kernel = kernel_rng.normal(0, 0.5, (3, 3)).astype(np.float32)
random_response = cv2.filter2D(feature_input, cv2.CV_32F, random_kernel,
                               borderType=cv2.BORDER_REPLICATE)
response_limit = max(0.001, float(np.abs(random_response).max()))
weight_limit = max(0.001, float(np.abs(random_kernel).max()))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(feature_input, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input: grayscale / 255')
axes[1].imshow(random_kernel, cmap='coolwarm', vmin=-weight_limit, vmax=weight_limit)
axes[1].set_title('Random kernel 3x3')
for row in range(3):
    for col in range(3):
        axes[1].text(col, row, f'{random_kernel[row, col]:.2f}', ha='center', va='center')
response_plot = axes[2].imshow(random_response, cmap='coolwarm',
                                vmin=-response_limit, vmax=response_limit)
axes[2].set_title('Output: signed response')
fig.colorbar(response_plot, ax=axes[2], fraction=0.046, pad=0.04)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
## สุ่ม 4 Kernel ขนาด 3x3 และแสดงผลลัพธ์ของ Feature Map ทั้ง 4 แบบ
# เริ่ม Seed เดิม: Kernel 1 จึงตรงกับตัวอย่างหนึ่ง Kernel
kernel_rng = np.random.default_rng(12)
random_kernels = kernel_rng.normal(0, 0.5, (4, 3, 3)).astype(np.float32)
feature_maps = np.zeros((4, filter_gray.shape[0], filter_gray.shape[1]), dtype=np.float32)
for i in range(4):
    feature_maps[i] = cv2.filter2D(feature_input, cv2.CV_32F, random_kernels[i],
                                  borderType=cv2.BORDER_REPLICATE)

map_limit = max(0.001, float(np.abs(feature_maps).max()))
kernel_limit = max(0.001, float(np.abs(random_kernels).max()))
fig, axes = plt.subplots(2, 4, figsize=(14, 8), layout='constrained')
for i in range(4):
    axes[0, i].imshow(random_kernels[i], cmap='coolwarm',
                       vmin=-kernel_limit, vmax=kernel_limit)
    axes[0, i].set_title(f'Random kernel {i + 1}')
    for row in range(3):
        for col in range(3):
            axes[0, i].text(col, row, f'{random_kernels[i, row, col]:.2f}',
                             ha='center', va='center')
    map_plot = axes[1, i].imshow(feature_maps[i], cmap='coolwarm',
                                  vmin=-map_limit, vmax=map_limit)
    axes[1, i].set_title(f'Feature map {i + 1}')
    axes[0, i].axis('off')
    axes[1, i].axis('off')
fig.colorbar(map_plot, ax=axes[1, :].tolist(), shrink=0.8, label='Signed response: shared scale')
plt.show()

## 11. Image Operations

คำนวณระหว่างพิกเซลตำแหน่งเดียวกันของหลายภาพ ต่างจาก Spatial Filtering ที่ใช้พิกเซลรอบข้าง โดยภาพต้องมีขนาดและ Channels ตรงกัน และหากเปรียบเทียบฉากเดิมต้องจัดตำแหน่งภาพให้ตรงกันด้วย


### 11.1 Image Subtraction

ลบภาพสองภาพเพื่อหาบริเวณที่เปลี่ยน เช่น เปรียบเทียบภาพฉากหลังกับภาพที่มีวัตถุ

$$g(x,y)=f(x,y)-h(x,y)$$

![Image Subtraction: examples of background subtraction](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/ImageSubstraction.png)

เมื่อต้องการดูขนาดความต่างโดยไม่สนใจเครื่องหมาย ใช้ $d=|f-h|$ หรือ `cv2.absdiff()` แล้วทำ Threshold เพื่อสร้าง Mask ของบริเวณที่เปลี่ยน แสงและเงาที่เปลี่ยนก็ทำให้เกิดความต่างได้ ไม่ได้หมายถึงวัตถุเคลื่อนที่เสมอไป


### 11.2 Image Averaging

เฉลี่ยภาพฉากเดิมหลายภาพเพื่อลด Noise โดยเฉลี่ยพิกเซลตำแหน่งเดียวกันข้ามภาพ ไม่ใช่เฉลี่ยเพื่อนบ้านแบบ Mean Filter

$$
g_i(x,y)=f(x,y)+n_i(x,y),\qquad
\bar g(x,y)=\frac{1}{K}\sum_{i=1}^{K}g_i(x,y)
$$

![Image Averaging: averaging multiple noisy images](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/PlushImage.png)

$K$ คือจำนวนภาพ เมื่อ Noise มีค่าเฉลี่ยเป็นศูนย์และไม่สัมพันธ์กันระหว่างภาพ การเฉลี่ยช่วยลดความแปรปรวนของ Noise ภาพต้องตรงกัน ไม่เช่นนั้นอาจเกิดภาพซ้อนหรือ Blur


In [ ]:
# ภาพเดียวกันที่เติม Noise ต่างกัน
noisy_images = []
for i in range(4):
    image = cv2.imread(str(image_dir / f'grace_noisy_{i + 1}.png'), 0)
    noisy_images.append(image)
average_image = np.mean(noisy_images, axis=0)  

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
images = [noisy_images[0], noisy_images[1], average_image]
titles = ['Noisy image 1', 'Noisy image 2', 'Average of 4 noisy images']
for i in range(3):
    axes[i].imshow(images[i], cmap='gray', vmin=0, vmax=255)
    axes[i].set_title(titles[i])
    axes[i].axis('off')
plt.tight_layout()
plt.show()


### 11.3 Logical Operations and Masking

ใช้ Mask เลือกบริเวณที่สนใจ (ROI) โดย Mask แบบ 8-bit ใช้ 255 เป็นส่วนที่เลือก และ 0 เป็นส่วนที่ไม่เลือก

$$I_{ROI}=I\;\mathrm{AND}\;M$$

![Logical AND and OR operations for masking](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W09_ImageProcessing/images/AndOrImage.png)

- **AND:** ภาพ AND กับ Mask จะคงภาพเฉพาะส่วนที่ Mask เป็น 255 ส่วนอื่นเป็นดำ ใช้ `cv2.bitwise_and()`
- **OR:** เมื่อนำ Binary Masks สองอันมา OR จะเลือกบริเวณที่อยู่ใน Mask ใดก็ได้ ใช้ `cv2.bitwise_or()`

ในภาพประกอบแถวล่าง การ OR ภาพกับ Mask ที่เป็น 255 นอก ROI ทำให้บริเวณนอก ROI เป็นขาว ส่วนที่ Mask เป็น 0 ยังคงภาพเดิม การทำ Bitwise OR ไม่ใช่การบวกความเข้มภาพ


In [ ]:
mask_image = cv2.imread(str(image_dir / 'grace_hopper.jpg'), 0)
h, w = mask_image.shape

# สอง Mask: เลือกส่วนบนและส่วนล่างของบุคคล
mask1 = np.zeros_like(mask_image)
mask2 = np.zeros_like(mask_image)
mask1[h//5:3*h//5, w//4:3*w//4] = 255
mask2[3*h//5:9*h//10, w//4:3*w//4] = 255
combined_mask = cv2.bitwise_or(mask1, mask2)  # รวมบริเวณที่เลือก
masked_image = cv2.bitwise_and(mask_image, combined_mask)  # เก็บภาพเฉพาะ ROI

fig, axes = plt.subplots(1, 4, figsize=(13, 4))
images = [mask1, mask2, combined_mask, masked_image]
titles = ['Mask 1', 'Mask 2', 'Mask 1 OR Mask 2', 'Image AND combined mask']
for i in range(4):
    axes[i].imshow(images[i], cmap='gray', vmin=0, vmax=255)
    axes[i].set_title(titles[i])
    axes[i].axis('off')
plt.tight_layout()
plt.show()


### 11.4 Weighted Addition: Vintage Texture


In [ ]:
portrait = cv2.imread(str(image_dir / 'grace_hopper.jpg'))
paper = cv2.imread(str(image_dir / 'vintage_paper.png'))
paper = cv2.resize(paper, (portrait.shape[1], portrait.shape[0]))
#vintage = cv2.addWeighted(portrait, 0.7, paper, 0.3, 0)
#vintage = portrait * 0.7 + paper * 0.3
vintage = (portrait).astype(np.float32) + paper.astype(np.float32)

# Check type
vintage = np.clip(vintage, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
images = [portrait, paper, vintage]
titles = ['Original', 'Old paper texture', 'original + texture']
for i in range(3):
    axes[i].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
    axes[i].set_title(titles[i])
    axes[i].axis('off')
plt.tight_layout()
plt.show()